# Local-Level Flow Computation (Milton + Helene)

**Purpose**: Compute within, inflow, and outflow time series at the local level
(per county for Milton, per cluster for Helene) and export as CSVs.

**No SARIMAX fitting here** — baselines are fitted in `local_baseline.ipynb`.

**Flow definitions** (see CLAUDE.md Section 4.2):
- **Within (self-loop)**: trips where both origin AND destination are in the local unit
  - Single county j: M[t, j, j] — internal trips within county j
  - Cluster C: sum of M[t, d, o] for all d,o in C — internal trips within cluster
- **Outflow**: trips FROM local unit TO outside A
- **Inflow**: trips FROM outside A TO local unit

**Outputs**: `results/local_level/{hurricane}/flow_ts_{within,outflow,inflow}.csv`

In [6]:
import pandas as pd
import numpy as np
import os
import sys
import datetime as dt
import warnings
from importlib import reload

import matplotlib.pyplot as plt

folder_path = './../../hurricane_oct/'
sys.path.append(folder_path)
sys.path.append(os.path.join(folder_path, 'mobility_function'))
from mobility_function import analysis as ma
ma = reload(ma)

%run ./recovery_function_v2.py

warnings.filterwarnings('ignore')

OUTPUT_MILTON = '../results/local_level/milton'
OUTPUT_HELENE = '../results/local_level/helene'
os.makedirs(os.path.join(OUTPUT_MILTON, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_HELENE, 'figures'), exist_ok=True)

print('Setup complete.')

Setup complete.


In [7]:
# ── Date setup ──
def mondays_str(year, start_month=7, end_month=10):
    start = dt.date(year, start_month, 28)
    end = dt.date(year, end_month, 31)
    days_ahead = (0 - start.weekday()) % 7
    cur = start + dt.timedelta(days=days_ahead)
    out = []
    while cur <= end:
        out.append(cur.strftime('%Y%m%d'))
        cur += dt.timedelta(days=7)
    return out

mondays_2023 = mondays_str(2023)
mondays_2024 = mondays_str(2024)
all_mondays = mondays_2023 + mondays_2024

dates_2023 = pd.date_range(start='2023-07-31', periods=len(mondays_2023)*7, freq='D')
dates_2024 = pd.date_range(start='2024-07-29', periods=len(mondays_2024)*7, freq='D')
dates_all = dates_2023.union(dates_2024)

# County index mapping
geo_idx = pd.read_csv('geoid_idx_names.csv')
geo_idx['GEOID'] = geo_idx['GEOID'].astype(int)
total_counties = geo_idx['county_idx'].max() + 1  # total counties in M tensor

print(f'Mondays: {len(all_mondays)}, Total days: {len(dates_all)}')
print(f'Total counties in tensor: {total_counties}')

Mondays: 28, Total days: 196
Total counties in tensor: 3144


In [ ]:
def compute_local_flows(M_sum, local_idx, all_A_idx):
    """
    Compute within (self-loop), inflow, outflow for a local unit.
    
    Parameters:
    -----------
    M_sum : ndarray, shape (days, dest, origin) — category-summed mobility tensor
    local_idx : array of int — county indices belonging to this local unit
    all_A_idx : array of int — ALL county indices in the affected region A
    
    Returns:
    --------
    within : ndarray (days,) — self-loop trips (both endpoints in local unit)
        Single county: M[d, j, j]
        Cluster: sum M[d, d_i, o_i] for all d_i, o_i in cluster
    outflow : ndarray (days,) — trips FROM local TO outside A
    inflow : ndarray (days,) — trips FROM outside A TO local
    """
    n_days = M_sum.shape[0]
    n_total = M_sum.shape[1]  # total counties dimension
    
    # Masks
    A_mask = np.zeros(n_total, dtype=bool)
    A_mask[all_A_idx] = True
    outside_A_mask = ~A_mask
    
    # ── WITHIN (self-loop): both endpoints in local unit ──
    if len(local_idx) == 1:
        # Single county: M_sum[:, j, j] — scalar indexing
        j = local_idx[0]
        within = M_sum[:, j, j]  # (days,)
    else:
        # Cluster: M_sum[:, c_idx, :][:, :, c_idx] then sum
        # This selects M[d, d_i, o_i] where BOTH d_i and o_i are in the cluster
        within = M_sum[:, local_idx, :][:, :, local_idx].sum(axis=(1, 2))  # (days,)
    
    # ── OUTFLOW: trips FROM local TO outside A ──
    # Total outgoing from local to all destinations
    outgoing_all = M_sum[:, :, local_idx].sum(axis=2)  # (days, all_dest)
    # Outflow = outgoing to outside-A destinations
    outflow = outgoing_all[:, outside_A_mask].sum(axis=1)  # (days,)
    
    # ── INFLOW: trips FROM outside A TO local ──
    incoming_all = M_sum[:, local_idx, :].sum(axis=1)  # (days, all_origin)
    # Inflow = incoming from outside-A origins
    inflow = incoming_all[:, outside_A_mask].sum(axis=1)  # (days,)
    
    return within, outflow, inflow

## 1. Milton — County-Level (N=21)

In [ ]:
# ── Load Milton affected counties ──
with open(f'{OUTPUT_MILTON}/counties_geoid_cut_50.txt', 'r') as f:
    milton_geoids = [int(line.strip()) for line in f]

milton_counties = geo_idx[geo_idx['GEOID'].isin(milton_geoids)].copy()
milton_counties = milton_counties.sort_values('county_idx').reset_index(drop=True)
all_A_idx_milton = milton_counties['county_idx'].values

print(f'Milton: {len(all_A_idx_milton)} counties in A')
print(f'County indices: {all_A_idx_milton}')

In [ ]:
# ── Compute all 3 flows for each Milton county ──
n_m = len(all_A_idx_milton)
m_within_ts = {i: [] for i in range(n_m)}
m_outflow_ts = {i: [] for i in range(n_m)}
m_inflow_ts = {i: [] for i in range(n_m)}

for date_str in all_mondays:
    print(f'  {date_str}...', end=' ', flush=True)
    M = ma.h5py_to_4d_array(folder_path + f'data/mobility/M_raw_{date_str}.h5')
    M_sum = M.sum(axis=1)  # (days, dest, origin)
    
    for j_id in range(n_m):
        local_idx = np.array([all_A_idx_milton[j_id]])  # single county
        w, o, i = compute_local_flows(M_sum, local_idx, all_A_idx_milton)
        m_within_ts[j_id].append(w)
        m_outflow_ts[j_id].append(o)
        m_inflow_ts[j_id].append(i)
    
    print('done')

# Concatenate
for j_id in range(n_m):
    m_within_ts[j_id] = np.concatenate(m_within_ts[j_id])
    m_outflow_ts[j_id] = np.concatenate(m_outflow_ts[j_id])
    m_inflow_ts[j_id] = np.concatenate(m_inflow_ts[j_id])

# Sanity check
print(f'\nSanity check (all values should be non-negative):')
for j_id in range(n_m):
    geoid = int(milton_counties.iloc[j_id]['GEOID'])
    name = milton_counties.iloc[j_id]['NAME']
    w_min = m_within_ts[j_id].min()
    o_min = m_outflow_ts[j_id].min()
    i_min = m_inflow_ts[j_id].min()
    flag = ' ⚠' if min(w_min, o_min, i_min) < 0 else ''
    print(f'  {name} ({geoid}): within={m_within_ts[j_id].mean():.0f}, '
          f'outflow={m_outflow_ts[j_id].mean():.0f}, '
          f'inflow={m_inflow_ts[j_id].mean():.0f}{flag}')

In [ ]:
# ── Export Milton raw flow time series ──
# Save as CSV: one row per day, columns = county GEOIDs

for flow_name, flow_ts in [('within', m_within_ts), ('outflow', m_outflow_ts), ('inflow', m_inflow_ts)]:
    # Build DataFrame: rows=dates, cols=counties
    data = {}
    for j_id in range(n_m):
        geoid = int(milton_counties.iloc[j_id]['GEOID'])
        data[str(geoid)] = flow_ts[j_id]
    
    df_export = pd.DataFrame(data, index=dates_all)
    df_export.index.name = 'date'
    csv_path = f'{OUTPUT_MILTON}/flow_ts_{flow_name}.csv'
    df_export.to_csv(csv_path)
    print(f'  Saved: {csv_path} ({df_export.shape})')

# Also export county metadata
milton_counties[['GEOID', 'county_idx', 'NAME']].to_csv(
    f'{OUTPUT_MILTON}/county_metadata.csv', index=False)
print(f'  Saved: {OUTPUT_MILTON}/county_metadata.csv')

## 2. Helene — Cluster-Level

In [ ]:
# ── Load Helene cluster assignments ──
cluster_df = pd.read_csv(f'{OUTPUT_HELENE}/county_cluster_assignments.csv')
cluster_df['GEOID'] = cluster_df['GEOID'].astype(int)
cluster_df = cluster_df.merge(geo_idx[['GEOID', 'county_idx']], on='GEOID', how='left')
assert cluster_df['county_idx'].notna().all(), 'Some counties missing from geo_idx!'
cluster_df['county_idx'] = cluster_df['county_idx'].astype(int)

# ALL affected counties in A (the full Helene region)
all_A_idx_helene = cluster_df['county_idx'].values

# Cluster → county indices
cluster_indices = {}
for c, grp in cluster_df.groupby('cluster'):
    cluster_indices[c] = grp['county_idx'].values

print(f'Helene: {len(cluster_indices)} clusters, {len(all_A_idx_helene)} counties in A')

In [ ]:
# ── Compute all 3 flows for each Helene cluster ──
# Key difference from before: within = from cluster to ALL of A, not just within-cluster

h_within_ts = {c: [] for c in cluster_indices}
h_outflow_ts = {c: [] for c in cluster_indices}
h_inflow_ts = {c: [] for c in cluster_indices}

for date_str in all_mondays:
    print(f'  {date_str}...', end=' ', flush=True)
    M = ma.h5py_to_4d_array(folder_path + f'data/mobility/M_raw_{date_str}.h5')
    M_sum = M.sum(axis=1)  # (days, dest, origin)
    
    for c, c_idx in cluster_indices.items():
        w, o, i = compute_local_flows(M_sum, c_idx, all_A_idx_helene)
        h_within_ts[c].append(w)
        h_outflow_ts[c].append(o)
        h_inflow_ts[c].append(i)
    
    print('done')

# Concatenate
for c in cluster_indices:
    h_within_ts[c] = np.concatenate(h_within_ts[c])
    h_outflow_ts[c] = np.concatenate(h_outflow_ts[c])
    h_inflow_ts[c] = np.concatenate(h_inflow_ts[c])

# Sanity check
print(f'\nSanity check (first 5 clusters):')
for c in sorted(cluster_indices.keys())[:5]:
    n_c = len(cluster_indices[c])
    w_min = h_within_ts[c].min()
    o_min = h_outflow_ts[c].min()
    i_min = h_inflow_ts[c].min()
    flag = ' ⚠' if min(w_min, o_min, i_min) < 0 else ''
    print(f'  Cluster {c} ({n_c} counties): within={h_within_ts[c].mean():.0f}, '
          f'outflow={h_outflow_ts[c].mean():.0f}, '
          f'inflow={h_inflow_ts[c].mean():.0f}{flag}')

In [ ]:
# ── Export Helene raw flow time series ──
for flow_name, flow_ts in [('within', h_within_ts), ('outflow', h_outflow_ts), ('inflow', h_inflow_ts)]:
    data = {}
    for c in sorted(cluster_indices.keys()):
        data[str(c)] = flow_ts[c]
    
    df_export = pd.DataFrame(data, index=dates_all)
    df_export.index.name = 'date'
    csv_path = f'{OUTPUT_HELENE}/flow_ts_{flow_name}.csv'
    df_export.to_csv(csv_path)
    print(f'  Saved: {csv_path} ({df_export.shape})')

print(f'  Cluster assignments: {OUTPUT_HELENE}/county_cluster_assignments.csv')

## 3. Summary

Raw flow time series saved. Next steps:
1. `local_baseline.ipynb` — fit SARIMAX on saved flows, export baselines
2. `local_metrics.ipynb` — compute metrics from baselines
3. `local_regression.ipynb` — OLS, quartile analysis, mixed model

In [ ]:
print('='*60)
print('FLOW COMPUTATION COMPLETE')
print('='*60)

print(f'\nMilton ({n_m} counties):')
for fn in ['within', 'outflow', 'inflow']:
    p = f'{OUTPUT_MILTON}/flow_ts_{fn}.csv'
    if os.path.exists(p):
        print(f'  {fn}: {p}')

print(f'\nHelene ({len(cluster_indices)} clusters):')
for fn in ['within', 'outflow', 'inflow']:
    p = f'{OUTPUT_HELENE}/flow_ts_{fn}.csv'
    if os.path.exists(p):
        print(f'  {fn}: {p}')